In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
safetensors_path = "/content/drive/MyDrive/model.safetensors"

開始

In [3]:
!pip install safetensors

In [4]:
!unzip -o /content/layer_00.zip -d /content/layer_00

Archive:  /content/layer_00.zip
  inflating: /content/layer_00/__MACOSX/._layer_00  
  inflating: /content/layer_00/layer_00/gate_up_weight_pc07.bin  
  inflating: /content/layer_00/__MACOSX/layer_00/._gate_up_weight_pc07.bin  
  inflating: /content/layer_00/layer_00/down_weight_slr0_pc26.bin  
  inflating: /content/layer_00/__MACOSX/layer_00/._down_weight_slr0_pc26.bin  
  inflating: /content/layer_00/layer_00/gate_up_weight_pc13.bin  
  inflating: /content/layer_00/__MACOSX/layer_00/._gate_up_weight_pc13.bin  
  inflating: /content/layer_00/layer_00/down_weight_slr1_pc26.bin  
  inflating: /content/layer_00/__MACOSX/layer_00/._down_weight_slr1_pc26.bin  
  inflating: /content/layer_00/layer_00/gate_up_scale_pc22.bin  
  inflating: /content/layer_00/__MACOSX/layer_00/._gate_up_scale_pc22.bin  
  inflating: /content/layer_00/layer_00/gate_up_scale_pc23.bin  
  inflating: /content/layer_00/__MACOSX/layer_00/._gate_up_scale_pc23.bin  
  inflating: /content/layer_00/layer_00/down_weight_s

第一步:讀取模組——把bin檔正確還原成golden model需要的格式

In [6]:
import numpy as np
import json
import os

# ============================================================
# 直接沿用同學程式碼裡,已經被verify_one_layer驗證過的函式
# 這兩個函式,負責把bin檔裡的原始bytes,正確還原成有號整數權重
# ============================================================

def decode_already_flipped(data_u8, num_bits):
    """
    這是同學程式碼裡的函式,原封不動搬過來使用
    負責:先處理bit打包(一個byte塞幾個數值),再做offset-binary轉two's complement
    """
    factor = 8 // num_bits
    mask = (1 << num_bits) - 1
    sign_bit = 1 << (num_bits - 1)
    out = np.empty(data_u8.shape[:-1] + (data_u8.shape[-1] * factor,), dtype=np.int8)
    for i in range(factor):
        raw = (data_u8.astype(np.int32) >> (i * num_bits)) & mask
        signed = np.where(raw >= sign_bit, raw - (1 << num_bits), raw).astype(np.int8)
        out[..., i::factor] = signed
    return out


def offset_binary_to_twos_complement_mask(num_bits):
    """算出magic這個XOR用的常數,同樣是搬同學的函式"""
    factor = 8 // num_bits
    mask = 0
    for i in range(factor):
        mask |= (1 << (i * num_bits + (num_bits - 1)))
    return mask


MAGIC = {nb: offset_binary_to_twos_complement_mask(nb) for nb in (2, 4)}


def load_layer_bin(layer_dir, num_bits, F, hidden_size=1536, NPC=32, DN=512):
    """
    讀取單一層的所有bin檔,還原成golden model要用的格式
    回傳:W_gate, W_up, W_down_slrs(list of 3), S_gate, S_up, S_down, act_scales
    """
    magic = MAGIC[num_bits]
    factor = 8 // num_bits
    BPC = hidden_size // factor          # 每個channel佔幾個bytes(打包後)
    N = 2 * F                            # gate+up合併後的總channel數
    CPP = DN // NPC                      # 每個HBM channel,一個tile裡佔幾個channel

    # ---- 讀取 gate_up weight,32個pc檔案接回來 ----
    hbm_streams = []
    for p in range(NPC):
        path = os.path.join(layer_dir, f"gate_up_weight_pc{p:02d}.bin")
        hbm_streams.append(np.fromfile(path, dtype=np.uint8))

    # 依照同學打包時的tile順序,把32個pc串流,重新排列回 [N, BPC] 的矩陣
    n_tiles = N // DN
    fused_flipped = np.empty((N, BPC), dtype=np.uint8)
    for t in range(n_tiles):
        for p in range(NPC):
            offset = t * CPP * BPC
            seg = hbm_streams[p][offset : offset + CPP * BPC].reshape(CPP, BPC)
            fused_flipped[t*DN + p*CPP : t*DN + (p+1)*CPP] = seg

    # 打包時做過 fused ^ magic,這裡讀回來要先XOR回去,才能送進decode
    fused_raw = fused_flipped ^ magic
    fused_int = decode_already_flipped(fused_raw, num_bits)  # 這裡才是真正的-8~7或-2~1整數值

    W_gate = fused_int[0::2, :]   # 偶數是gate
    W_up   = fused_int[1::2, :]   # 奇數是up

    # ---- 讀取 gate_up scale ----
    scale_streams = []
    for p in range(NPC):
        path = os.path.join(layer_dir, f"gate_up_scale_pc{p:02d}.bin")
        scale_streams.append(np.fromfile(path, dtype=np.float32))

    combined_scale = np.empty((N,), dtype=np.float32)
    for t in range(n_tiles):
        for p in range(NPC):
            offset = t * CPP
            seg = scale_streams[p][offset : offset + CPP]
            combined_scale[t*DN + p*CPP : t*DN + (p+1)*CPP] = seg

    S_gate = combined_scale[0::2]
    S_up   = combined_scale[1::2]

    # ---- 讀取 down_proj weight(3個SLR,各自32個pc)----
    K_down = F
    BPC_FULL = K_down // factor
    SEG = BPC_FULL // 3
    n_lane = 128
    CPP_D = n_lane // NPC
    n_tiles_down = hidden_size // DN
    n_passes = DN // n_lane

    W_down_slrs = []
    for s in range(3):
        pc_streams = []
        for p in range(NPC):
            path = os.path.join(layer_dir, f"down_weight_slr{s}_pc{p:02d}.bin")
            pc_streams.append(np.fromfile(path, dtype=np.uint8))

        seg_flipped = np.empty((hidden_size, SEG), dtype=np.uint8)
        for t in range(n_tiles_down):
            for q in range(n_passes):
                for p in range(NPC):
                    offset = (t * n_passes + q) * CPP_D * SEG
                    blk = pc_streams[p][offset : offset + CPP_D * SEG].reshape(CPP_D, SEG)
                    row_base = t * DN + q * n_lane + p * CPP_D
                    seg_flipped[row_base : row_base + CPP_D] = blk

        seg_raw = seg_flipped ^ magic
        seg_int = decode_already_flipped(seg_raw, num_bits)  # shape [hidden_size, SEG*factor]
        W_down_slrs.append(seg_int)

    # ---- 讀取 down_proj scale(3份一樣,取一份即可)----
    S_down = np.fromfile(os.path.join(layer_dir, "down_scale_slr0.bin"), dtype=np.float32)

    # ---- 讀取 activation scales ----
    with open(os.path.join(layer_dir, "activation_scales.json")) as f:
        act_scales = json.load(f)

    return W_gate, W_up, W_down_slrs, S_gate, S_up, S_down, act_scales

第二步:驗證——不要一寫完讀取程式碼就直接接去跑MLP,先驗證讀出來的東西對不對
這一步至關重要,建議你先跟safetensors原始資料對照一次,而不是直接信任你剛寫的讀取程式碼。

In [7]:
from safetensors import safe_open

def verify_loaded_data(layer_dir, layer_idx, safetensors_path, num_bits, F):
    W_gate, W_up, W_down_slrs, S_gate, S_up, S_down, act_scales = load_layer_bin(layer_dir, num_bits, F)

    with safe_open(safetensors_path, framework="numpy") as f:
        prefix = f"model.language_model.layers.{layer_idx}.mlp"
        gate_u8_official = f.get_tensor(f"{prefix}.gate_proj.weight")
        S_gate_official = f.get_tensor(f"{prefix}.gate_proj.weight_scale")

        # 把官方原始的gate_u8,也用同一套decode流程還原一次,當作「正確答案」
        magic = MAGIC[num_bits]
        # official_int = decode_already_flipped(gate_u8_official ^ magic, num_bits) -> 錯誤（官方不需要做magic）
        official_int = decode_already_flipped(gate_u8_official , num_bits) #gemini教我這樣改且成功

    # 比對:從bin檔重組出來的W_gate,跟直接從safetensors還原出來的official_int,應該完全一致
    match_weight = np.array_equal(W_gate, official_int)
    match_scale = np.allclose(S_gate, S_gate_official.flatten())

    print(f"Layer {layer_idx} weight比對: {'✓ 通過' if match_weight else '✗ 不一致!'}")
    print(f"Layer {layer_idx} scale比對: {'✓ 通過' if match_scale else '✗ 不一致!'}")

    if not match_weight:
        diff_count = np.sum(W_gate != official_int)
        print(f"  不一致的元素數量: {diff_count} / {W_gate.size}")

    return match_weight and match_scale

第三步:接上你原本已經驗證過的MLP golden model

In [18]:
import numpy as np

# ============================================================
# 1. 共用基礎模組 (硬體實際在做的事)
# ============================================================

def mac_accumulate(x_int, W_int):
    acc = np.zeros(W_int.shape[0], dtype=np.int32)
    for k in range(len(x_int)):
        acc += W_int[:, k].astype(np.int32) * x_int[k].astype(np.int32)
    return acc

def requantize_m_k(Q_acc, S_eff, k_bits=16, out_bits=8):
    M = np.round(S_eff * (2 ** k_bits)).astype(np.int64)
    Temp = Q_acc.astype(np.int64) * M
    rounding_offset = 1 << (k_bits - 1)   # 加上"半格"的偏移量,四捨五入（rounding effect）
    shifted = (Temp + rounding_offset) >> k_bits

    if out_bits == 8:
        return np.clip(shifted, -128, 127).astype(np.int8)
    elif out_bits == 16:
        return np.clip(shifted, -32768, 32767).astype(np.int16)
    return shifted.astype(np.int32)

# ============================================================
# 2. [Offline] 離線產生 INT8 LUT
# ============================================================
def generate_gelu_lut(S_gate_out, S_gelu_out):
    lut = np.zeros(256, dtype=np.int8)
    for i in range(256):
        x_int8 = i - 256 if i >= 128 else i
        x_float = x_int8 * S_gate_out

        # 拆成兩行寫，避免括號數量算錯，同時解決 overflow 警告
        exp_val = np.exp(np.clip(-1.702 * x_float, -50.0, 50.0))
        gelu_float = x_float * (1.0 / (1.0 + exp_val))

        y_int8 = np.clip(np.round(gelu_float / S_gelu_out), -128, 127)
        lut[i] = np.int8(y_int8)
    return lut

def run_mlp_from_bin(layer_dir, layer_idx, x_int, S_x, num_bits, F, k_bits=16):
    W_gate, W_up, W_down_slrs, S_gate, S_up, S_down, act_scales = load_layer_bin(layer_dir, num_bits, F)

    S_gate_out = act_scales["gate_proj_output"]
    S_up_out = act_scales["up_proj_output"]
    S_down_in = act_scales["down_proj_input"]

    # S_gelu_out 依然要靠你自己校準,manifest/bin檔裡沒有這個值
    # S_gelu_out = 0.1   # TODO: 換成你校準過的真實數值
    S_gelu_out = 0.294718
    gelu_lut = generate_gelu_lut(S_gate_out, S_gelu_out)

    # ---- gate_proj ----
    gate_acc = mac_accumulate(x_int, W_gate)
    S_eff_gate = (S_x * S_gate.flatten()) / S_gate_out
    gate_int8 = requantize_m_k(gate_acc, S_eff_gate, k_bits, out_bits=8)
    lut_indices = gate_int8.astype(np.int32) & 0xFF
    gate_activated_int8 = gelu_lut[lut_indices]

    # ---- up_proj ----
    up_acc = mac_accumulate(x_int, W_up)
    S_eff_up = (S_x * S_up.flatten()) / S_up_out
    up_int8 = requantize_m_k(up_acc, S_eff_up, k_bits, out_bits=8)

    # ---- 相乘 ----
    activated_int16 = gate_activated_int8.astype(np.int16) * up_int8.astype(np.int16)
    S_eff_mult = (S_gelu_out * S_up_out) / S_down_in
    down_in_int8 = requantize_m_k(activated_int16, S_eff_mult, k_bits, out_bits=8)

    # ---- down_proj:三段式,對應三個SLR,各自MAC再加總 ----
    factor = 8 // num_bits
    SEG_elements = W_down_slrs[0].shape[1]   # 每段涵蓋的K方向元素數(已經是decode後的維度)

    partial_accs = []
    for s in range(3):
        x_slice = down_in_int8[s*SEG_elements : (s+1)*SEG_elements]
        partial_acc = mac_accumulate(x_slice, W_down_slrs[s])
        partial_accs.append(partial_acc)

    down_acc = sum(partial_accs)   # 跨SLR加總
    down_float = down_acc.astype(np.float32) * S_down_in * S_down.flatten()

    return down_float

In [19]:
layer_dir = "/content/layer_00/layer_00"   # 依實際解壓縮後的路徑調整,如果裡面還有一層資料夾要對應改

# 先跑驗證
verify_loaded_data(layer_dir, layer_idx=0, safetensors_path=safetensors_path, num_bits=4, F=6144)

Layer 0 weight比對: ✓ 通過
Layer 0 scale比對: ✓ 通過


True

以下是gemini改的

In [20]:
# 1. 輸入浮點數還原 (保留)
x_float = x_int.astype(np.float32) * S_x

# 2. 權重浮點數還原 (使用 .reshape(-1, 1) 來進行正確的列廣播)
W_gate_float = W_gate.astype(np.float32) * S_gate.reshape(-1, 1)
W_up_float   = W_up.astype(np.float32) * S_up.reshape(-1, 1)

# Down Proj: 將 3 個 SLR 沿著 K 軸 (axis=1) 拼回 (1536, 6144) 矩陣，再乘上 Scale
W_down_int   = np.concatenate(W_down_slrs, axis=1)
W_down_float = W_down_int.astype(np.float32) * S_down.reshape(-1, 1)

In [21]:
import numpy as np

# ============================================================
# 1. 補上 Missing 的參考函式定義 (FP Reference & Evaluation)
# ============================================================
def mlp_fp16_reference(x_float, W_gate_float, W_up_float, W_down_float):
    """使用原生浮點數矩陣計算 FP Baseline"""
    gate_out = np.dot(W_gate_float, x_float)
    up_out = np.dot(W_up_float, x_float)

    # GeLU 激活函數
    exp_val = np.exp(np.clip(-1.702 * gate_out, -50.0, 50.0))
    gelu_out = gate_out * (1.0 / (1.0 + exp_val))

    # Element-wise 相乘與 Down Projection
    mult_out = gelu_out * up_out
    down_out = np.dot(W_down_float, mult_out)
    return down_out

def evaluate_quantization_accuracy(fp_output, quant_output):
    """計算 Cosine Similarity 與 MAE / Relative RMSE"""
    dot_product = np.dot(fp_output, quant_output)
    norm_fp = np.linalg.norm(fp_output)
    norm_quant = np.linalg.norm(quant_output)
    cos_sim = dot_product / (norm_fp * norm_quant + 1e-8)

    mae = np.mean(np.abs(fp_output - quant_output))
    rmse = np.sqrt(np.mean((fp_output - quant_output) ** 2))
    rel_rmse = rmse / (norm_fp / np.sqrt(len(fp_output)) + 1e-8)

    print("\n========= 🎯 Phase 2 最終對齊結果 (Accuracy Report) =========")
    print(f"1. Cosine Similarity (餘弦相似度) : {cos_sim:.6f}  (目標 > 0.98)")
    print(f"2. Mean Absolute Error (MAE 平均誤差): {mae:.6f}")
    print(f"3. Relative RMSE (相對均方根誤差)   : {rel_rmse:.6f}")

    if cos_sim >= 0.98:
        print("\n🎉 恭喜！Golden Model 完美對齊！可以開始產出 .hex 測試檔給硬體同學了！")
    elif cos_sim >= 0.90:
        print("\n⚠️ 相似度尚可，但有些許偏差。建議微調 S_gelu_out 以符合實際動態範圍。")
    else:
        print("\n❌ 相似度偏低，請檢查矩陣方向或量化 Scale 設定。")

# ============================================================
# 2. 執行主程式測試
# ============================================================
# 1. 準備模擬輸入
np.random.seed(42)
x_int = np.random.randint(-128, 128, size=1536).astype(np.int8)
S_x = 0.05
x_float = x_int.astype(np.float32) * S_x

# 2. 設定參數與路徑
layer_dir = "/content/layer_00/layer_00"
layer_idx = 0
num_bits = 4
F = 6144

# 3. 執行新版 Golden Model
down_float_quant = run_mlp_from_bin(
    layer_dir, layer_idx, x_int, S_x, num_bits, F
)

# 4. 建立 FP16 Baseline
W_gate, W_up, W_down_slrs, S_gate, S_up, S_down, _ = load_layer_bin(
    layer_dir, num_bits, F
)

W_gate_float = W_gate.astype(np.float32) * S_gate.reshape(-1, 1)
W_up_float   = W_up.astype(np.float32) * S_up.reshape(-1, 1)

W_down_int   = np.concatenate(W_down_slrs, axis=1)
W_down_float = W_down_int.astype(np.float32) * S_down.reshape(-1, 1)

# 5. 計算原生全浮點數基準值
fp_output = mlp_fp16_reference(x_float, W_gate_float, W_up_float, W_down_float)

# 6. 印出前 5 個數值與進行精度比對
print("原生 FP16 前 5 個輸出 :", fp_output[:5])
print("INT8 LUT 量化前 5 個輸出:", down_float_quant[:5])

evaluate_quantization_accuracy(fp_output, down_float_quant)

原生 FP16 前 5 個輸出 : [394.81238  216.83531  171.43748  -11.121185 180.19905 ]
INT8 LUT 量化前 5 個輸出: [404.07358   232.9161    169.75883    -3.3644967 182.41858  ]

========= 🎯 Phase 2 最終對齊結果 (Accuracy Report) =========
1. Cosine Similarity (餘弦相似度) : 0.996842  (目標 > 0.98)
2. Mean Absolute Error (MAE 平均誤差): 9.836925
3. Relative RMSE (相對均方根誤差)   : 0.079807

🎉 恭喜！Golden Model 完美對齊！可以開始產出 .hex 測試檔給硬體同學了！
